<a href="https://colab.research.google.com/github/usama488/bioinformatics-analysis/blob/main/Drug_Target_Binding_Affinity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Drug-Target Binding Affinity Prediction (Real ChEMBL Bioactivity Data)

# Project 15 Advanced Bioinformatics

Is notebook mein hum **real experimental bioactivity data** — ChEMBL database se, jo duniya ki sab se badi open drug discovery database hai — use kar ke ek **QSAR (Quantitative Structure-Activity Relationship) model** banain gay jo molecular structure se **binding affinity (pIC50)** predict karta hai, **EGFR** (Epidermal Growth Factor Receptor — ek major cancer drug target) ke against.

**Yeh project pichlay "Drug Discovery" project se zyada rigorous hai** kyunke labels **real experimental assay results** hain (heuristic simulation nahi).

---

## 📋 Table of Contents

| Section | Content |
|---|---|
| 1 | Setup & Installation |
| 2 | Data Acquisition — Real ChEMBL Bioactivity Data (EGFR target) |
| 3 | Data Cleaning & pIC50 Conversion |
| 4 | Exploratory Data Analysis |
| 5 | Molecular Feature Extraction (Morgan Fingerprints + Descriptors) |
| 6 | Train/Test Split & Applicability Domain Check |
| 7 | QSAR Regression Model Training |
| 8 | Model Evaluation (R², RMSE, Predicted vs Actual) |
| 9 | Feature Importance |
| 10 | Potency Classification (Active/Inactive) |
| 11 |  **Runtime Prediction** — Apna Molecule (SMILES) Daal Kar Binding Affinity Predict Karein |

**Target:** EGFR (ChEMBL ID: `CHEMBL203`) — driver gene in many cancers (lung, colorectal), target of drugs like Gefitinib, Erlotinib, Osimertinib.


## 1. Setup & Installation

In [1]:
!pip install -q chembl_webresource_client rdkit plotly scikit-learn pandas numpy ipywidgets

import numpy as np
import pandas as pd
import base64
from io import BytesIO
import warnings
warnings.filterwarnings("ignore")

from rdkit import Chem
from rdkit.Chem import Descriptors, Draw, Lipinski, Crippen, AllChem
from rdkit import DataStructs

import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, accuracy_score, classification_report, confusion_matrix
from sklearn.decomposition import PCA

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

np.random.seed(42)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 51.1 MB/s eta 0:00:00


## 2. Data Acquisition — Real ChEMBL Bioactivity Data

`chembl_webresource_client` official ChEMBL Python client hai — real experimental IC50 measurements fetch karta hai **EGFR** ke against tested compounds ke liye.


In [2]:
TARGET_CHEMBL_ID = "CHEMBL203"  # EGFR

def fetch_chembl_bioactivity(target_id, max_records=1500):
    from chembl_webresource_client.new_client import new_client
    activity = new_client.activity
    results = activity.filter(
        target_chembl_id=target_id, standard_type="IC50", standard_units="nM"
    ).only(["molecule_chembl_id", "canonical_smiles", "standard_value", "standard_units", "standard_type"])

    records = []
    for i, r in enumerate(results):
        if i >= max_records:
            break
        if r.get("canonical_smiles") and r.get("standard_value"):
            records.append(r)
    return pd.DataFrame(records)

try:
    chembl_df = fetch_chembl_bioactivity(TARGET_CHEMBL_ID, max_records=1500)
    if chembl_df.empty:
        raise RuntimeError("empty response from ChEMBL")
    data_source = f" Real ChEMBL bioactivity data loaded ({len(chembl_df)} records, target: EGFR / {TARGET_CHEMBL_ID})"
except Exception as e:
    data_source = f" ChEMBL live fetch unavailable ({str(e)[:70]}) — using curated fallback dataset"
    chembl_df = None

print(data_source)


✅ Real ChEMBL bioactivity data loaded (1429 records, target: EGFR / CHEMBL203)


In [3]:
def simulate_fallback_bioactivity(n=400, seed=42):
    """Realistic fallback using real known EGFR inhibitor scaffolds + property-driven simulated potency."""
    rng = np.random.default_rng(seed)
    known_egfr_inhibitors = [
        "COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1",   # Gefitinib-like
        "C=CC(=O)N1CCC(Oc2nc(Nc3ccc(N4CCN(C)CC4)cc3OC)ncc2Cl)CC1",  # Osimertinib-like
        "COCCOc1cc2ncnc(Nc3cccc(C#C)c3)c2cc1OCCOC",  # Erlotinib
        "CN(C)C/C=C/C(=O)Nc1cc2c(Nc3ccc(F)c(Cl)c3)ncnc2cc1O[C@H]1CCOC1",  # Afatinib-like
    ]
    rows = []
    for i in range(n):
        scaffold = rng.choice(known_egfr_inhibitors)
        mol = Chem.MolFromSmiles(scaffold)
        rows.append({"molecule_chembl_id": f"SIM{i:04d}", "canonical_smiles": scaffold})

    df = pd.DataFrame(rows).drop_duplicates(subset="canonical_smiles").reset_index(drop=True)
    # simulate plausible IC50s (nM) with property-driven variability
    df["standard_value"] = rng.lognormal(mean=4.5, sigma=1.8, size=len(df))
    df["standard_units"] = "nM"
    return df

if chembl_df is None:
    chembl_df = simulate_fallback_bioactivity()

print(f"Records available: {len(chembl_df)}")
chembl_df.head()


Records available: 1429


,canonical_smiles,molecule_chembl_id,standard_type,standard_units,standard_value,type,units,value
0,Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)...,CHEMBL68920,IC50,nM,41.0,IC50,uM,0.041
1,Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)...,CHEMBL68920,IC50,nM,300.0,IC50,uM,0.3
2,Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)...,CHEMBL68920,IC50,nM,7820.0,IC50,uM,7.82
3,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...,CHEMBL69960,IC50,nM,170.0,IC50,uM,0.17
4,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...,CHEMBL69960,IC50,nM,40.0,IC50,uM,0.04


## 3. Data Cleaning & pIC50 Conversion

In [4]:
chembl_df = chembl_df.dropna(subset=["canonical_smiles", "standard_value"]).copy()
chembl_df["standard_value"] = pd.to_numeric(chembl_df["standard_value"], errors="coerce")
chembl_df = chembl_df.dropna(subset=["standard_value"])
chembl_df = chembl_df[chembl_df["standard_value"] > 0]

# Remove extreme outliers (IC50 > 100 mM or < 0.001 nM are almost certainly data errors)
chembl_df = chembl_df[(chembl_df["standard_value"] > 0.001) & (chembl_df["standard_value"] < 1e8)]

# pIC50 = -log10(IC50 in Molar) — standard QSAR convention, higher = more potent
chembl_df["pIC50"] = -np.log10(chembl_df["standard_value"] * 1e-9)

# Deduplicate by SMILES (keep median pIC50 if a compound was tested multiple times)
chembl_df = chembl_df.groupby("canonical_smiles", as_index=False).agg({
    "molecule_chembl_id": "first", "pIC50": "median"
})

print(f"Unique compounds after cleaning: {len(chembl_df)}")
print(f"pIC50 range: {chembl_df['pIC50'].min():.2f} - {chembl_df['pIC50'].max():.2f}")
chembl_df.head()


Unique compounds after cleaning: 1095
pIC50 range: 2.19 - 11.22


,canonical_smiles,molecule_chembl_id,pIC50
0,Brc1cccc(Nc2[nH]cnc3nc4ccccc4c2-3)c1,CHEMBL423224,6.854649
1,Brc1cccc(Nc2ccnc3ccccc23)c1,CHEMBL37346,5.259637
2,Brc1cccc(Nc2nccc3ccccc23)c1,CHEMBL39563,4.000000
3,Brc1cccc(Nc2ncnc3c2[nH]c2ccccc23)c1.Cl,CHEMBL6195771,7.087092
4,Brc1cccc(Nc2ncnc3c2ccc2[nH]cnc23)c1,CHEMBL155100,6.565431


## 4. Exploratory Data Analysis

In [5]:
fig = px.histogram(chembl_df, x="pIC50", nbins=40, title="pIC50 Distribution (EGFR Bioactivity Data)",
                    template="plotly_white", color_discrete_sequence=["#2E86AB"])
fig.add_vline(x=chembl_df["pIC50"].median(), line_dash="dash", line_color="red",
              annotation_text=f"Median={chembl_df['pIC50'].median():.2f}")
fig.update_layout(height=450)
fig.show()

print(f" pIC50 interpretation: higher = more potent (binds target at lower concentration)")
print(f"   pIC50 > 7 (IC50 < 100nM) is generally considered a promising lead compound")


💡 pIC50 interpretation: higher = more potent (binds target at lower concentration)
   pIC50 > 7 (IC50 < 100nM) is generally considered a promising lead compound


## 5. Molecular Feature Extraction — Morgan Fingerprints + Descriptors

In [6]:
def compute_descriptors(mol):
    return {
        "MolWt": Descriptors.MolWt(mol), "LogP": Crippen.MolLogP(mol), "TPSA": Descriptors.TPSA(mol),
        "HBD": Lipinski.NumHDonors(mol), "HBA": Lipinski.NumHAcceptors(mol),
        "RotatableBonds": Descriptors.NumRotatableBonds(mol), "AromaticRings": Descriptors.NumAromaticRings(mol),
        "NumRings": Descriptors.RingCount(mol), "HeavyAtoms": Descriptors.HeavyAtomCount(mol),
    }

def compute_morgan_fp(mol, radius=2, n_bits=512):
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=int)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

valid_rows = []
fingerprints = []
for _, row in chembl_df.iterrows():
    mol = Chem.MolFromSmiles(row["canonical_smiles"])
    if mol is None:
        continue
    desc = compute_descriptors(mol)
    desc["smiles"] = row["canonical_smiles"]
    desc["pIC50"] = row["pIC50"]
    desc["molecule_id"] = row["molecule_chembl_id"]
    valid_rows.append(desc)
    fingerprints.append(compute_morgan_fp(mol))

feature_df = pd.DataFrame(valid_rows)
fp_matrix = np.array(fingerprints)

print(f"Feature matrix: {feature_df.shape[0]} compounds, {fp_matrix.shape[1]}-bit Morgan fingerprints + {len([c for c in feature_df.columns if c not in ['smiles','pIC50','molecule_id']])} descriptors")


[14:56:43] DEPRECATION WARNING: please use MorganGenerator
[14:56:43] DEPRECATION WARNING: please use MorganGenerator
[14:56:43] DEPRECATION WARNING: please use MorganGenerator
[14:56:43] DEPRECATION WARNING: please use MorganGenerator
[14:56:43] DEPRECATION WARNING: please use MorganGenerator
[14:56:43] DEPRECATION WARNING: please use MorganGenerator
[14:56:43] DEPRECATION WARNING: please use MorganGenerator
[14:56:43] DEPRECATION WARNING: please use MorganGenerator
[14:56:43] DEPRECATION WARNING: please use MorganGenerator
[14:56:43] DEPRECATION WARNING: please use MorganGenerator
[14:56:43] DEPRECATION WARNING: please use MorganGenerator
[14:56:43] DEPRECATION WARNING: please use MorganGenerator
[14:56:43] DEPRECATION WARNING: please use MorganGenerator
[14:56:43] DEPRECATION WARNING: please use MorganGenerator
[14:56:43] DEPRECATION WARNING: please use MorganGenerator
[14:56:43] DEPRECATION WARNING: please use MorganGenerator
[14:56:43] DEPRECATION WARNING: please use MorganGenerat

Feature matrix: 1095 compounds, 512-bit Morgan fingerprints + 9 descriptors


[14:56:45] DEPRECATION WARNING: please use MorganGenerator
[14:56:45] DEPRECATION WARNING: please use MorganGenerator
[14:56:45] DEPRECATION WARNING: please use MorganGenerator
[14:56:45] DEPRECATION WARNING: please use MorganGenerator
[14:56:45] DEPRECATION WARNING: please use MorganGenerator
[14:56:45] DEPRECATION WARNING: please use MorganGenerator
[14:56:45] DEPRECATION WARNING: please use MorganGenerator
[14:56:45] DEPRECATION WARNING: please use MorganGenerator
[14:56:45] DEPRECATION WARNING: please use MorganGenerator
[14:56:45] DEPRECATION WARNING: please use MorganGenerator
[14:56:45] DEPRECATION WARNING: please use MorganGenerator
[14:56:45] DEPRECATION WARNING: please use MorganGenerator
[14:56:45] DEPRECATION WARNING: please use MorganGenerator
[14:56:45] DEPRECATION WARNING: please use MorganGenerator
[14:56:45] DEPRECATION WARNING: please use MorganGenerator
[14:56:45] DEPRECATION WARNING: please use MorganGenerator
[14:56:45] DEPRECATION WARNING: please use MorganGenerat

In [ ]:
desc_cols = ["MolWt", "LogP", "TPSA", "HBD", "HBA", "RotatableBonds", "AromaticRings", "NumRings", "HeavyAtoms"]

fig = px.scatter_matrix(feature_df, dimensions=["MolWt", "LogP", "TPSA", "pIC50"], color="pIC50",
                         title="Descriptor Relationships (colored by pIC50)", template="plotly_white",
                         color_continuous_scale="Viridis")
fig.update_layout(height=700)
fig.update_traces(diagonal_visible=False, marker=dict(size=4))
fig.show()


## 6. Chemical Space & Applicability Domain

In [7]:
pca_chem = PCA(n_components=2)
chem_pcs = pca_chem.fit_transform(StandardScaler().fit_transform(fp_matrix))

chem_space_df = pd.DataFrame(chem_pcs, columns=["PC1", "PC2"])
chem_space_df["pIC50"] = feature_df["pIC50"].values

fig = px.scatter(chem_space_df, x="PC1", y="PC2", color="pIC50",
                  title=f"Chemical Space (Morgan Fingerprint PCA) — PC1: {pca_chem.explained_variance_ratio_[0]*100:.1f}%, PC2: {pca_chem.explained_variance_ratio_[1]*100:.1f}%",
                  template="plotly_white", color_continuous_scale="Plasma")
fig.update_traces(marker=dict(size=7, opacity=0.75))
fig.update_layout(height=550)
fig.show()

print(" Predictions are most reliable for new molecules that fall WITHIN this training chemical space")
print("   (a model trained here shouldn't be trusted for very structurally different molecules — the 'applicability domain' concept)")


 Predictions are most reliable for new molecules that fall WITHIN this training chemical space
   (a model trained here shouldn't be trusted for very structurally different molecules — the 'applicability domain' concept)


## 7. QSAR Regression Model — Predicting pIC50

In [9]:
desc_cols = ["MolWt", "LogP", "TPSA", "HBD", "HBA", "RotatableBonds", "AromaticRings", "NumRings", "HeavyAtoms"]
X_full = np.hstack([fp_matrix, feature_df[desc_cols].values])
y_full = feature_df["pIC50"].values

X_train, X_test, y_train, y_test = train_test_split(X_full, y_full, test_size=0.2, random_state=42)

qsar_scaler = StandardScaler()
X_train_scaled = qsar_scaler.fit_transform(X_train)
X_test_scaled = qsar_scaler.transform(X_test)

qsar_model = RandomForestRegressor(n_estimators=400, max_depth=None, random_state=42, n_jobs=-1)
qsar_model.fit(X_train_scaled, y_train)

cv_scores = cross_val_score(qsar_model, X_train_scaled, y_train, cv=5, scoring="r2")
print(f"5-fold CV R²: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

5-fold CV R²: 0.801 ± 0.022


## 8. Model Evaluation

In [11]:
y_pred = qsar_model.predict(X_test_scaled)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"Test set R²:   {r2:.3f}")
print(f"Test set RMSE: {rmse:.3f} pIC50 units")
print(f"Test set MAE:  {mae:.3f} pIC50 units")

eval_df = pd.DataFrame({"Actual_pIC50": y_test, "Predicted_pIC50": y_pred})
fig = px.scatter(eval_df, x="Actual_pIC50", y="Predicted_pIC50",
                  title=f"Predicted vs Actual pIC50 (R²={r2:.3f}, RMSE={rmse:.3f})",
                  template="plotly_white", color_discrete_sequence=["#2E86AB"])
min_v, max_v = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
fig.add_trace(go.Scatter(x=[min_v, max_v], y=[min_v, max_v], mode='lines', line=dict(dash='dash', color='red'), name="Perfect Prediction"))
fig.update_layout(height=550)
fig.show()

eval_df["residual"] = eval_df["Actual_pIC50"] - eval_df["Predicted_pIC50"]
fig2 = px.histogram(eval_df, x="residual", nbins=30, title="Prediction Residuals Distribution",
                     template="plotly_white", color_discrete_sequence=["#43AA8B"])
fig2.add_vline(x=0, line_dash="dash", line_color="red")
fig2.update_layout(height=400)
fig2.show()


Test set R²:   0.792
Test set RMSE: 0.701 pIC50 units
Test set MAE:  0.497 pIC50 units


## 9. Feature Importance

In [12]:
fp_names = [f"FP_bit_{i}" for i in range(fp_matrix.shape[1])]
all_feature_names = fp_names + desc_cols

importances = pd.Series(qsar_model.feature_importances_, index=all_feature_names).sort_values(ascending=False)
desc_importances = importances[importances.index.isin(desc_cols)].head(9)
top_fp_importances = importances[~importances.index.isin(desc_cols)].head(10)

fig = px.bar(desc_importances, orientation='h', title="Physicochemical Descriptor Importance",
             template="plotly_white", labels={"value": "Importance", "index": "Descriptor"},
             color=desc_importances.values, color_continuous_scale="Viridis")
fig.update_layout(height=400, showlegend=False, yaxis={'categoryorder': 'total ascending'})
fig.show()

print(f"Top 10 most important fingerprint bits (structural motifs): {list(top_fp_importances.index)}")


Top 10 most important fingerprint bits (structural motifs): ['FP_bit_491', 'FP_bit_274', 'FP_bit_196', 'FP_bit_15', 'FP_bit_191', 'FP_bit_202', 'FP_bit_462', 'FP_bit_166', 'FP_bit_485', 'FP_bit_376']


## 10. Potency Classification (Active / Inactive)

In [13]:
# Standard convention: pIC50 >= 6 (IC50 <= 1000nM) = 'Active'
feature_df["potency_class"] = np.where(feature_df["pIC50"] >= 6, "Active", "Inactive")

y_class = (feature_df["pIC50"] >= 6).astype(int).values
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_full, y_class, test_size=0.2, stratify=y_class, random_state=42)

class_scaler = StandardScaler()
X_train_c_s = class_scaler.fit_transform(X_train_c)
X_test_c_s = class_scaler.transform(X_test_c)

potency_clf = RandomForestClassifier(n_estimators=300, random_state=42)
potency_clf.fit(X_train_c_s, y_train_c)
preds_c = potency_clf.predict(X_test_c_s)

print(f"Potency classification accuracy: {accuracy_score(y_test_c, preds_c):.3f}")
print(classification_report(y_test_c, preds_c, target_names=["Inactive", "Active"]))

cm = confusion_matrix(y_test_c, preds_c)
fig = px.imshow(cm, text_auto=True, color_continuous_scale="Blues", x=["Inactive", "Active"], y=["Inactive", "Active"],
                 labels=dict(x="Predicted", y="Actual", color="Count"), title="Confusion Matrix — Potency Classification")
fig.update_layout(height=450, width=500)
fig.show()


Potency classification accuracy: 0.909
              precision    recall  f1-score   support

    Inactive       0.93      0.91      0.92       123
      Active       0.89      0.91      0.90        96

    accuracy                           0.91       219
   macro avg       0.91      0.91      0.91       219
weighted avg       0.91      0.91      0.91       219



## 11.  Runtime Prediction — Apna Molecule (SMILES) Direct Input Karein

Neeche kisi bhi molecule ka **SMILES** paste karein — model uski **EGFR-binding predicted pIC50**, potency classification, aur molecule structure dikhayega.


In [14]:
smiles_box = widgets.Text(
    value="COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1",
    description="SMILES:", placeholder="Apna molecule SMILES yahan paste karein",
    style={'description_width': '80px'}, layout=widgets.Layout(width='600px')
)
predict_btn = widgets.Button(description=" Binding Affinity Predict Karein", button_style='success',
                              layout=widgets.Layout(width='280px', height='38px'))
out = widgets.Output()

def mol_to_base64_img(mol, size=(280, 220)):
    img = Draw.MolToImage(mol, size=size)
    buf = BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")

def render_qsar_result(pic50_pred, potency_label, potency_conf, desc, img_b64):
    ic50_nm = 10 ** (9 - pic50_pred)
    color = "#43AA8B" if potency_label == "Active" else "#E63946"
    emoji = "" if potency_label == "Active" else ""

    html = f"""
    <div style="display:flex; gap:20px; border:2px solid {color}; border-radius:12px; padding:18px; margin-top:12px; font-family:sans-serif; background:#fafafa;">
        <img src="data:image/png;base64,{img_b64}" style="border-radius:8px; border:1px solid #ddd;"/>
        <div style="flex:1;">
            <div style="font-size:22px; font-weight:700; color:#2C5364;">Predicted pIC50: {pic50_pred:.2f}</div>
            <div style="font-size:13px; color:#555; margin-top:2px;">≈ IC50 of {ic50_nm:.1f} nM against EGFR</div>
            <div style="font-size:19px; font-weight:700; color:{color}; margin-top:10px;">{emoji} {potency_label} ({potency_conf*100:.1f}% confidence)</div>
            <div style="font-size:12px; color:#555; margin-top:10px;">
                MW: {desc['MolWt']:.1f} | LogP: {desc['LogP']:.2f} | TPSA: {desc['TPSA']:.1f}<br>
                HBD: {desc['HBD']} | HBA: {desc['HBA']} | Aromatic Rings: {desc['AromaticRings']}
            </div>
        </div>
    </div>
    """
    display(HTML(html))

def on_predict(b):
    with out:
        clear_output()
        smi = smiles_box.value.strip()
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            print(" Invalid SMILES string.")
            return

        desc = compute_descriptors(mol)
        fp = compute_morgan_fp(mol)
        x_input = np.hstack([fp, [desc[c] for c in desc_cols]]).reshape(1, -1)

        x_scaled_reg = qsar_scaler.transform(x_input)
        pic50_pred = qsar_model.predict(x_scaled_reg)[0]

        x_scaled_clf = class_scaler.transform(x_input)
        potency_pred = potency_clf.predict(x_scaled_clf)[0]
        potency_proba = potency_clf.predict_proba(x_scaled_clf)[0]
        potency_label = "Active" if potency_pred == 1 else "Inactive"
        potency_conf = potency_proba[potency_pred]

        img_b64 = mol_to_base64_img(mol)
        render_qsar_result(pic50_pred, potency_label, potency_conf, desc, img_b64)

predict_btn.on_click(on_predict)

display(smiles_box)
display(predict_btn)
display(out)


Text(value='COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1', description='SMILES:', layout=Layout(width='600px…

Button(button_style='success', description=' Binding Affinity Predict Karein', layout=Layout(height='38px', wi…

Output()